# SICD Open-Source Test — Gemma 4 31B (A100 Optimized)

This notebook runs the **Semantic Interference** experiment using a local **Gemma 4 31B** model on a Colab A100 GPU.

**Why Gemma 4 31B?**
- Frontier reasoning capabilities released 3 days ago.
- 31B parameters allow for deep biomedical understanding.
- Running locally ensures 100% data privacy and zero API costs.

**Requirements:**
- Colab A100 80GB Runtime.
- Hugging Face Token (with Read permissions).

In [ ]:
# ── 0. Environment & Model Setup ──────────────────────────────────────────────
import os, sys, shutil, torch
from pathlib import Path

# 1. Install necessary libraries for local inference
# NOTE: We install transformers from source because Gemma 4 is very new!
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/huggingface/transformers.git
    !pip install -q accelerate bitsandbytes sentencepiece protobuf

# 2. Recursive Path Detection
def setup_environment():
    found_root = None
    for root, dirs, files in os.walk('/content'):
        if 'sicd_cases.py' in files:
            found_root = root
            break
    if not found_root and 'cot_generator.py' in os.listdir('/content'):
        found_root = '/content'

    if found_root:
        os.chdir(found_root)
        if found_root not in sys.path: sys.path.insert(0, found_root)
        if not os.path.exists('utils'): os.makedirs('utils', exist_ok=True)
        utils_files = ['cot_generator.py', 'concept_extractor_v2.py', 'umls_api_linker.py', 
                       'umls_checker.py', 'umls_density_scorer.py', 'umls_specificity_scorer.py', 
                       'umls_oscillation_scorer.py', 'umls_regression_scorer.py', 'umls_local_db.py']
        for f in utils_files:
            if os.path.exists(f) and not os.path.isdir(f): shutil.move(f, os.path.join('utils', f))
        if not os.path.exists('utils/__init__.py'): 
            with open('utils/__init__.py', 'w') as f: pass
        print(f'Environment configured at: {found_root}')
    else:
        print("WARNING: Experiment files not found. Please upload sicd_cases.py etc.")

setup_environment()
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

In [ ]:
# ── 1. Configuration (Keys & Token) ───────────────────────────────────────────
os.environ['HF_TOKEN'] = 'REDACTED'
os.environ['UMLS_API_KEY'] = 'REDACTED'
print('Config ready.')

In [ ]:
# ── 2. Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from utils.cot_generator import generate as cot_generate
from utils.concept_extractor_v2 import extract_concepts
from utils.umls_density_scorer import score_density
from utils.umls_specificity_scorer import score_specificity
from utils.umls_oscillation_scorer import score_oscillation
from utils.umls_regression_scorer import score_regression
from utils import umls_api_linker

from sicd_cases import CASES, INTERFERENCE_LEVELS, build_prompt
from sicd_cases import DOMAIN_SEMANTIC_TYPES, DOMAIN_KEYWORDS
from sicd_scorers import score_split_density

print('All modules imported.')
print(f'UMLS Linked: {umls_api_linker.is_configured()}')

In [ ]:
# ── 3. Model Warming ──────────────────────────────────────────────────────────
print("Warming up Gemma 4 31B (this may take a few minutes)...")
warmup_result = cot_generate("Quick test: what is medical drift?", prefer='local', model='google/gemma-4-31B-it')
print(f"\nModel ready. Response length: {len(warmup_result['steps'])} steps.")

In [ ]:
# ── 4. CoT Generation (40 chains) ─────────────────────────────────────────────
CACHE_PATH = Path('data/sicd_hf_cache.json')
Path('data').mkdir(exist_ok=True)

if CACHE_PATH.exists():
    print('Loading from cache...')
    raw_chains = json.loads(CACHE_PATH.read_text())
else:
    print('Generating SICD chains with Gemma 4 31B...')
    raw_chains = {}
    for case in CASES:
        raw_chains[case['id']] = {}
        for label, ordinal in INTERFERENCE_LEVELS:
            prompt = build_prompt(case, label)
            result = cot_generate(prompt, prefer='local', model='google/gemma-4-31B-it', temperature=1.0)
            raw_chains[case['id']][label] = result['steps']
            print(f'  {case["title"]} [{label}] -> {len(result["steps"])} steps')
    CACHE_PATH.write_text(json.dumps(raw_chains, indent=2))

CHAINS = []
for ci, case in enumerate(CASES):
    for label, ordinal in INTERFERENCE_LEVELS:
        steps = raw_chains[case['id']][label]
        CHAINS.append({
            'id': f'{case["id"]}_{label}', 'label': f'{case["title"]} - {label}',
            'steps': steps, 'case_idx': ci, 'level_label': label, 'gt_ordinal': ordinal,
            'target_domain': case['domain'], 'interference_domain': case['interference_domain'],
        })
print(f'\n{len(CHAINS)} chains generated.')

In [ ]:
# ── 5. Concept Extraction & Scoring ───────────────────────────────────────────
print('Processing chains (UMLS Extraction + CGD Scoring)...')
for i, chain in enumerate(CHAINS):
    chain['concepts'] = extract_concepts(chain['steps'], top_k=5)
    chain['density']     = score_density(chain['concepts'], steps=chain['steps'])
    chain['specificity'] = score_specificity(chain['concepts'])
    chain['oscillation'] = score_oscillation(chain['concepts'])
    chain['regression']  = score_regression(chain['concepts'])
    chain['split_density'] = score_split_density(
        chain['concepts'], 
        target_domain=chain['target_domain'], 
        interference_domain=chain['interference_domain'],
        domain_semantic_types=DOMAIN_SEMANTIC_TYPES, 
        domain_keywords=DOMAIN_KEYWORDS,
        steps=chain['steps']
    )
    if (i+1) % 5 == 0: print(f'  Done {i+1}/{len(CHAINS)}...')
print('Scoring complete.')

In [ ]:
# ── 6. Evaluation ─────────────────────────────────────────────────────────────
rows = []
for ch in CHAINS:
    rows.append({
        'GT Ordinal': ch['gt_ordinal'],
        'Den Slope':  ch['density'].get('slope', 0.0) or 0.0,
        'Spec Slope': ch['specificity'].get('depth_slope', 0.0) or 0.0,
        'Osc Score':  ch['oscillation'].get('oscillation_score', 0.0),
        'Reg Score':  ch['regression'].get('regression_score', 0.0),
        'Split Ratio': ch['split_density'].get('mean_density_ratio', 1.0),
    })
df = pd.DataFrame(rows)

print('\n=== SICD Results (Local Gemma 4 31B) ===')
print(f'{"Signal":<25} {"Spearman rho":>12} {"p-value":>10}  Result')
print('-' * 65)

signal_cols = [('Density slope', 'Den Slope', 1), ('Specificity slope', 'Spec Slope', 1),
               ('Oscillation', 'Osc Score', 1), ('Regression', 'Reg Score', 1),
               ('Split Ratio', 'Split Ratio', -1)]

for name, col, expected_dir in signal_cols:
    rho, pval = spearmanr(df[col], df['GT Ordinal'])
    confirmed = (rho * expected_dir) > 0
    tag = 'CONFIRMED' if (confirmed and pval < 0.05) else ('REVERSED' if not confirmed else 'NOT SIG')
    print(f'{name:<25} {rho:+12.3f} {pval:10.4f}  {tag:>10} {"*" if pval < 0.05 else ""}')

In [ ]:
# ── 7. Visualization ──────────────────────────────────────────────────────────
level_tick = ['Control', 'Soft', 'Hard', 'Dissonance']
sig_defs = [('Density Slope', 'density', 'slope'), ('Spec Slope', 'specificity', 'depth_slope'),
            ('Oscillation', 'oscillation', 'oscillation_score'), ('Regression', 'regression', 'regression_score'),
            ('Split Ratio', 'split_density', 'mean_density_ratio')]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (sig_name, sig_key, score_key) in zip(axes, sig_defs):
    means = [np.mean([ch[sig_key].get(score_key, 0.0) or 0.0 for ch in CHAINS if ch['gt_ordinal'] == oi]) for oi in range(4)]
    ax.plot(range(4), means, 'b-s', linewidth=2, markersize=8)
    ax.set_xticks(range(4)); ax.set_xticklabels(level_tick, fontsize=8); ax.set_title(sig_name); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()